# Install comfyUI and other packages

In [ ]:
!apt-get install -y wget aria2 libgl1-mesa-glx
!rm /content/sample_data

%cd /content/

!git clone https://github.com/comfyanonymous/ComfyUI.git
%cd ComfyUI
!pip install -r requirements.txt

# !pip install https://github.com/nunchaku-tech/nunchaku/releases/download/v0.3.1/nunchaku-0.3.1+torch2.7-cp311-cp311-linux_x86_64.whl

# Install coustom Node (comfyUI-gguf)

In [ ]:
%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/city96/ComfyUI-GGUF
!pip install --upgrade gguf

# Download gguff Models

In [ ]:
# #1. UNet GGUF
!wget -c "https://huggingface.co/QuantStack/Qwen-Image-Edit-2509-GGUF/resolve/main/Qwen-Image-Edit-2509-Q3_K_S.gguf" -O /content/ComfyUI/models/unet/Qwen-Image-Edit-2509-Q3_K_S.gguf
!wget -c "https://huggingface.co/QuantStack/Qwen-Image-Edit-2509-GGUF/resolve/main/Qwen-Image-Edit-2509-Q2_K.gguf" -O /content/ComfyUI/models/unet/Qwen-Image-Edit-2509-Q2_K.gguf

# 2. CLIP Text Encoder & Vision Projector
!wget -c "https://huggingface.co/unsloth/Qwen2.5-VL-7B-Instruct-GGUF/resolve/main/Qwen2.5-VL-7B-Instruct-Q3_K_S.gguf" -O /content/ComfyUI/models/text_encoders/Qwen2.5-VL-7B-Instruct-Q3_K_S.gguf
!wget -c "https://huggingface.co/Mungert/Qwen2.5-VL-7B-Instruct-GGUF/resolve/main/Qwen2.5-VL-7B-Instruct-mmproj-f16.gguf" -O /content/ComfyUI/models/text_encoders/Qwen2.5-VL-7B-Instruct-mmproj-F16.gguf

# 3. VAE
# !rm /content/ComfyUI/models/vae/qwen_image_vae.safetensors
!wget -c "https://huggingface.co/Comfy-Org/Qwen-Image_ComfyUI/resolve/main/split_files/vae/qwen_image_vae.safetensors" -O /content/ComfyUI/models/vae/qwen_image_vae.safetensors

# 4. Lightning LoRA
!wget -c "https://huggingface.co/lightx2v/Qwen-Image-Lightning/resolve/main/Qwen-Image-Lightning-4steps-V1.0-bf16.safetensors" -O /content/ComfyUI/models/loras/Qwen-Image-Lightning-4steps-V1.0-bf16.safetensors

print("All Qwen-Image-Edit GGUF files downloaded successfully!")

# Run comfyui on the background with pinggy

In [ ]:
!pip install pinggy -q

import os, subprocess, time, signal, psutil

# ── 0. Kill any previous ComfyUI & Pinggy instances ─────────
print("🧹 Cleaning up previous instances...")

killed = 0
for proc in psutil.process_iter(['pid', 'name', 'cmdline']):
    try:
        cmdline = ' '.join(proc.info['cmdline'] or [])
        if any(x in cmdline for x in ['main.py', 'pinggy', 'ComfyUI']):
            os.kill(proc.info['pid'], signal.SIGKILL)
            killed += 1
    except (psutil.NoSuchProcess, psutil.AccessDenied):
        pass

if killed:
    print(f"Killed {killed} leftover process(es)")
    time.sleep(2)  # let ports free up
else:
    print("⏭ No previous instances found")

# Also force-free port 8188 specifically
subprocess.run("fuser -k 8188/tcp 2>/dev/null || true", shell=True)

# ── 1. Install ComfyUI Manager ──────────────────────────────
manager_path = "/content/ComfyUI/custom_nodes/ComfyUI-Manager"

if not os.path.exists(manager_path):
    subprocess.run(["git", "clone", "https://github.com/ltdrdata/ComfyUI-Manager", manager_path], check=True)
    print("ComfyUI Manager installed")
else:
    print("⏭ Manager already installed")

# ── 2. Relax Manager security (required for tunnel URLs) ────
os.makedirs(manager_path, exist_ok=True)
with open(f"{manager_path}/config.ini", "w") as f:
    f.write("[default]\nsecurity_level = weak\n")
print("Manager security set to weak")

# ── 3. Start ComfyUI in background ──────────────────────────
comfy_proc = subprocess.Popen(
    ["python", "main.py", "--listen", "0.0.0.0", "--port", "8188"],
    cwd="/content/ComfyUI",
    stdout=subprocess.DEVNULL,  # suppress log spam
    stderr=subprocess.DEVNULL
)
time.sleep(8)

if comfy_proc.poll() is None:
    print(f" ComfyUI started (PID {comfy_proc.pid})")
else:
    print("❌ ComfyUI failed to start — check logs")

# ── 4. Start Pinggy tunnel ───────────────────────────────────
import pinggy
tunnel = pinggy.start_tunnel(forwardto="localhost:8188")
print(f"\n🌐 Open this URL: {tunnel.urls}")

# Running a prosses

In [ ]:
# running a prosses to keep the cell alive
import time
from IPython.display import display, HTML, clear_output

stop_clock = False

def time_clock():
    global stop_clock
    stop_clock = False
    while not stop_clock:
        current_time = time.strftime("%H:%M:%S")
        print(f"\rCurrent Time: {current_time}", end="")
        time.sleep(1)

time_clock()

